<a href="https://colab.research.google.com/github/jesusessu/MDD_LAB08/blob/develop/MDD_LAB08.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Minería de Datos - Semana 8: ÁRBOLES DE DECISIÓN. BOSQUES ALEATORIOS**

Alumno:

Esplana Sulla Jesús Zósimo

In [2]:
# Instalación de librerías necesarias (solo si no están instaladas)
!pip install -q ucimlrepo imbalanced-learn

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, accuracy_score, precision_score,
                             recall_score, f1_score)
from sklearn.impute import SimpleImputer

from imblearn.over_sampling import SMOTE
from ucimlrepo import fetch_ucirepo

warnings.filterwarnings('ignore')
sns.set(style='whitegrid')

In [4]:
# Cargar datos desde ucimlrepo
data = fetch_ucirepo(id=15)
df = pd.concat([data.data.features, data.data.targets], axis=1)
print(f'Dimensiones del dataset: {df.shape}')
df.head()

Dimensiones del dataset: (699, 10)


,Clump_thickness,Uniformity_of_cell_size,Uniformity_of_cell_shape,Marginal_adhesion,Single_epithelial_cell_size,Bare_nuclei,Bland_chromatin,Normal_nucleoli,Mitoses,Class
0,5,1,1,1,2,1.0,3,1,1,2
1,5,4,4,5,7,10.0,3,2,1,2
2,3,1,1,1,2,2.0,3,1,1,2
3,6,8,8,1,3,4.0,3,7,1,2
4,4,1,1,3,2,1.0,3,1,1,2


In [5]:
def preprocess_data(df, target):
    X = df.drop(columns=[target]).copy()
    y = df[target].copy()

    num_cols = X.select_dtypes(include=np.number).columns.tolist()

    if X.isnull().sum().sum() > 0:
        X[num_cols] = SimpleImputer(strategy='mean').fit_transform(X[num_cols])

    for col in num_cols:
        Q1, Q3 = X[col].quantile([0.25, 0.75])
        IQR = Q3 - Q1
        X[col] = X[col].clip(Q1 - 1.5*IQR, Q3 + 1.5*IQR)

    X[num_cols] = StandardScaler().fit_transform(X[num_cols])

    if y.value_counts(normalize=True).min() < 0.5:
        X, y = SMOTE(random_state=42).fit_resample(X, y)

    return X, y

In [6]:
X, y = preprocess_data(df, 'Class')
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
print(f'Tamaño entrenamiento: {X_train.shape}, prueba: {X_test.shape}')

Tamaño entrenamiento: (687, 9), prueba: (229, 9)


In [7]:
def train_decision_tree(X_train, y_train):
    grid = GridSearchCV(DecisionTreeClassifier(random_state=42),
                        {'max_depth': list(range(1, 11))},
                        cv=5, n_jobs=-1)
    grid.fit(X_train, y_train)
    return grid.best_estimator_

In [8]:
def train_random_forest(X_train, y_train):
    grid = GridSearchCV(RandomForestClassifier(random_state=42),
                        {'n_estimators': [50, 100, 200], 'max_depth': [None, 5, 10]},
                        cv=5, n_jobs=-1)
    grid.fit(X_train, y_train)
    return grid.best_estimator_

In [9]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred))
    return {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='weighted'),
        'recall': recall_score(y_test, y_pred, average='weighted'),
        'f1_score': f1_score(y_test, y_pred, average='weighted')
    }

In [10]:
# Árbol de Decisión
tree_model = train_decision_tree(X_train, y_train)
print(f"Profundidad óptima: {tree_model.get_depth()}")
tree_metrics = evaluate_model(tree_model, X_test, y_test)

Profundidad óptima: 5
              precision    recall  f1-score   support

           2       0.96      0.95      0.96       115
           4       0.95      0.96      0.96       114

    accuracy                           0.96       229
   macro avg       0.96      0.96      0.96       229
weighted avg       0.96      0.96      0.96       229



In [11]:
# Bosque Aleatorio
rf_model = train_random_forest(X_train, y_train)
print(f"Random Forest - Estimators: {rf_model.n_estimators}, Max Depth: {rf_model.max_depth}")
rf_metrics = evaluate_model(rf_model, X_test, y_test)

Random Forest - Estimators: 100, Max Depth: None
              precision    recall  f1-score   support

           2       0.99      0.97      0.98       115
           4       0.97      0.99      0.98       114

    accuracy                           0.98       229
   macro avg       0.98      0.98      0.98       229
weighted avg       0.98      0.98      0.98       229



In [12]:
# Comparación de resultados
print("\n--- Comparación de Accuracies ---")
print(f"Árbol de Decisión: {tree_metrics['accuracy']:.4f}")
print(f"Bosque Aleatorio: {rf_metrics['accuracy']:.4f}")


--- Comparación de Accuracies ---
Árbol de Decisión: 0.9563
Bosque Aleatorio: 0.9782
